# Training-scaling with MaxFreq: bias vs. effective dimension

This notebook reproduces the paper's MaxFreq scaling results. It compares the training-performance gap `Delta_{f-c}MSE_min = MSE_full_min - MSE_cut_min` and the normalized effective dimension (ED, Eq. 12) of a "full" (high-ED) partial Fourier model against a "cutoff" (lower-ED, decay-truncated) counterpart, as a function of the maximum Fourier input frequency MaxFreq. Results are compared for two data-generating regimes produced by the companion scripts `train_and_FIM_biased.py` (target function exactly reproducible by both models, i.e. biased) and `train_and_FIM_UNbiased.py` (generic/unrelated target function, i.e. unbiased), across several numbers of training points per feature (Ntrain). The notebook loads the pickled per-model-draw results, aggregates them across random model/data draws, and reproduces the paper's plot of `Delta_{f-c}MSE_min` against the input-basis dimension D = 2*MaxFreq - 1.

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
import re
import pickle

import pennylane.numpy as np



import matplotlib
import matplotlib.pyplot as plt



# Current path for importing custom functions
path_base = '/home/b/b309245/cloud_cover_qml_parametrization/DYAMOND_cellbased_implementations/'
sys.path.insert(0, path_base + 'qml_useful_functions')

import qnn_layouts_pennylane
importlib.reload(qnn_layouts_pennylane)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import training_functions_jax
importlib.reload(training_functions_jax)

Sets the experiment identifiers (results folder, biased/unbiased data-gen tags, learning rate, batch size, `cutoff`, `decay_exp`, `bond_dim`, Dloc, and the fixed `no_params`) and the scanned MaxFreq values (`max_frequency_vec`), matching those used when running `train_and_FIM_biased.py`/`train_and_FIM_UNbiased.py`, so the saved result filenames can be reconstructed and located.

In [ ]:
### ---------------------------------------------------------------------------------------- ###
## ------------------------------------ Basic model specs ----------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

# Folder in which to load data from
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/scaling_training_with_MaxFreq/'

name_data_gen_fullbiased = 'biased_data_gen'
name_data_gen_unbiased = 'UNbiased_data_gen'

# Learning rate
learning_rate = 0.02
learning_rate_name = '0p02'

# Batch size for training
batch_size = 5
batch_size_name = str(batch_size)

### Cutoff index for correlations among Fourier components
cutoff = 2
cutoff_name = str(cutoff)

### Decay exponent for cutoff model: 
### S_cut[cutoff+i-1] = np.exp(- decay_exp * i) for i in [1, dim_basis_inputs-cutoff]
decay_exp = 3.0
decay_exp_name = '3p0'

### No. of features (dimension of input vectors)
no_of_features = 1
name_no_features = str(no_of_features)

# Bond dimension
bond_dim = 120
name_bond_dim = str(bond_dim)

### No. basis states per parameter
dim_basis_single_param = 3  ### (1, cos(th), sin(th))
name_dim_basis_param = str(dim_basis_single_param)

### No. of parameters
no_params = 50
name_no_params = str(no_params)

### Max. freq.
max_frequency_vec = [4, 6, 8, 10, 12, 14, 16, 18, 20, 24, 28, 32, 36, 40, 44, 48, 54, 60]

Scans the results folder for pickled per-model-draw result files (`dict_results...pkl`) matching the biased data-gen filename pattern, to discover which values of Ntrain (training points per feature) are actually available on disk.

In [ ]:
name_0 = 'dict_results'
name_1 = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq')
name_2 = ('_Nparams' + name_no_params + '_DimBasisParam' + name_dim_basis_param + '_' + name_data_gen_fullbiased + 
          '_NtrainPerFeat')
name_3 = ('_batch' + batch_size_name + 
          '_lr' + learning_rate_name + '_cutoff' + cutoff_name + '_decayexp' + decay_exp_name + 
          '_modeldraw')

filename0 = name_0 + name_1
listallfiles = [f for f in os.listdir(results_folder) 
                if (f.startswith(filename0) and (('_' + name_data_gen_fullbiased) in f))]
all_n_train = []
for filename in listallfiles:
    res = re.findall((name_0 + name_1 + '(\S+)' + name_2 + '(\S+)' + name_3 + '(\S+).pkl'), filename)
    max_freq = res[0][0]
    max_freq = int(max_freq)
    n_train = res[0][1]
    n_train = int(n_train)
    nm = res[0][2]
    nm = int(nm)
    all_n_train.append(n_train)
all_n_train = np.asarray(all_n_train)
all_n_train = np.unique(all_n_train)
# all_n_train should be 20, 30, 40, 50
print(all_n_train)

For each available Ntrain value, aggregates the raw per-model-draw `dict_results...pkl` files across all random model draws, separately for the biased and unbiased data-generating cases, into per-MaxFreq dictionaries of `Delta_{f-c}MSE_min` statistics, MSE training trajectories, and normalized FIM spectra/effective dimension; the aggregated results are then pickled to `extracted_results_FULLbiased...`/`extracted_results_UNbiased...`.

In [ ]:
for n_train in all_n_train:
    name_0 = 'dict_results'
    name_1 = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq')
    name_2 = ('_Nparams' + name_no_params + '_DimBasisParam' + name_dim_basis_param + '_' + name_data_gen_fullbiased + 
              '_NtrainPerFeat' + str(n_train))
    name_3 = ('_batch' + batch_size_name + 
              '_lr' + learning_rate_name + '_cutoff' + cutoff_name + '_decayexp' + decay_exp_name + 
              '_modeldraw')
    filename0 = name_0 + name_1
    listallfiles = [f for f in os.listdir(results_folder) 
                    if (f.startswith(filename0) and (('_' + name_data_gen_fullbiased) in f) and (('_NtrainPerFeat' + str(n_train)) in f))]
    dict_fullbiased_all = dict()
    for filename in listallfiles:
        res = re.findall((name_0 + name_1 + '(\S+)' + name_2 + name_3 + '(\S+).pkl'), filename)
        max_freq = res[0][0]
        max_freq = int(max_freq)
        nm = res[0][1]
        nm = int(nm)
        dict_d = dict()
        dict_d['delta_minMSE_mean'] = []
        dict_d['delta_minMSE_std'] = []
        dict_d['delta_minMSE_min'] = []
        dict_d['delta_minMSE_max'] = []
        dict_d['MSE_full_mean'] = []
        dict_d['MSE_full_std'] = []
        dict_d['MSE_full_min'] = []
        dict_d['MSE_full_max'] = []
        dict_d['MSE_cut_mean'] = []
        dict_d['MSE_cut_std'] = []
        dict_d['MSE_cut_min'] = []
        dict_d['MSE_cut_max'] = []
        dict_d['mean_normFIM_spectra_full'] = []
        dict_d['std_normFIM_spectra_full'] = []
        dict_d['min_normFIM_spectra_full'] = []
        dict_d['max_normFIM_spectra_full'] = []
        dict_d['norm_eff_dim_full'] = []
        dict_d['mean_normFIM_spectra_cut'] = []
        dict_d['std_normFIM_spectra_cut'] = []
        dict_d['min_normFIM_spectra_cut'] = []
        dict_d['max_normFIM_spectra_cut'] = []
        dict_d['norm_eff_dim_cut'] = []
        dict_fullbiased_all[max_freq] = dict_d
    for filename in listallfiles:
        res = re.findall((name_0 + name_1 + '(\S+)' + name_2 + name_3 + '(\S+).pkl'), filename)
        max_freq = res[0][0]
        max_freq = int(max_freq)
        nm = res[0][1]
        nm = int(nm)
        path_file = os.path.join(results_folder, filename)
        with open(path_file, 'rb') as f:
            dict_model = pickle.load(f)
        dict_d = dict_fullbiased_all[max_freq]
        dict_d['delta_minMSE_mean'].append(np.mean(dict_model['delta_minMSE_full_m_cut']))
        dict_d['delta_minMSE_std'].append(np.std(dict_model['delta_minMSE_full_m_cut']))
        dict_d['delta_minMSE_min'].append(np.min(dict_model['delta_minMSE_full_m_cut']))
        dict_d['delta_minMSE_max'].append(np.max(dict_model['delta_minMSE_full_m_cut']))
        dict_d['MSE_full_mean'].append(dict_model['MSE_full_mean'])
        dict_d['MSE_full_std'].append(dict_model['MSE_full_std'])
        dict_d['MSE_full_min'].append(dict_model['MSE_full_min'])
        dict_d['MSE_full_max'].append(dict_model['MSE_full_max'])
        dict_d['MSE_cut_mean'].append(dict_model['MSE_cut_mean'])
        dict_d['MSE_cut_std'].append(dict_model['MSE_cut_std'])
        dict_d['MSE_cut_min'].append(dict_model['MSE_cut_min'])
        dict_d['MSE_cut_max'].append(dict_model['MSE_cut_max'])
        dict_d['mean_normFIM_spectra_full'].append(dict_model['mean_normFIM_spectra_full'])
        dict_d['std_normFIM_spectra_full'].append(dict_model['std_normFIM_spectra_full'])
        dict_d['min_normFIM_spectra_full'].append(dict_model['min_normFIM_spectra_full'])
        dict_d['max_normFIM_spectra_full'].append(dict_model['max_normFIM_spectra_full'])
        dict_d['norm_eff_dim_full'].append(dict_model['norm_eff_dim_full'])
        dict_d['mean_normFIM_spectra_cut'].append(dict_model['mean_normFIM_spectra_cut'])
        dict_d['std_normFIM_spectra_cut'].append(dict_model['std_normFIM_spectra_cut'])
        dict_d['min_normFIM_spectra_cut'].append(dict_model['min_normFIM_spectra_cut'])
        dict_d['max_normFIM_spectra_cut'].append(dict_model['max_normFIM_spectra_cut'])
        dict_d['norm_eff_dim_cut'].append(dict_model['norm_eff_dim_cut'])
        dict_fullbiased_all[max_freq] = dict_d
    print('Finished biased for Ntrain ', n_train)    

    name_0 = 'dict_results'
    name_1 = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + '_MaxFreq')
    name_2 = ('_Nparams' + name_no_params + '_DimBasisParam' + name_dim_basis_param + '_' + name_data_gen_unbiased + 
              '_NtrainPerFeat' + str(n_train))
    name_3 = ('_batch' + batch_size_name + 
              '_lr' + learning_rate_name + '_cutoff' + cutoff_name + '_decayexp' + decay_exp_name + 
              '_modeldraw')
    filename0 = name_0 + name_1
    listallfiles = [f for f in os.listdir(results_folder) 
                    if (f.startswith(filename0) and (('_' + name_data_gen_unbiased) in f) and (('_NtrainPerFeat' + str(n_train)) in f))]
    dict_unbiased_all = dict()
    for filename in listallfiles:
        res = re.findall((name_0 + name_1 + '(\S+)' + name_2 + name_3 + '(\S+).pkl'), filename)
        max_freq = res[0][0]
        max_freq = int(max_freq)
        nm = res[0][1]
        nm = int(nm)
        dict_d = dict()
        dict_d['delta_minMSE_mean'] = []
        dict_d['delta_minMSE_std'] = []
        dict_d['delta_minMSE_min'] = []
        dict_d['delta_minMSE_max'] = []
        dict_d['MSE_full_mean'] = []
        dict_d['MSE_full_std'] = []
        dict_d['MSE_full_min'] = []
        dict_d['MSE_full_max'] = []
        dict_d['MSE_cut_mean'] = []
        dict_d['MSE_cut_std'] = []
        dict_d['MSE_cut_min'] = []
        dict_d['MSE_cut_max'] = []
        dict_d['mean_normFIM_spectra_full'] = []
        dict_d['std_normFIM_spectra_full'] = []
        dict_d['min_normFIM_spectra_full'] = []
        dict_d['max_normFIM_spectra_full'] = []
        dict_d['norm_eff_dim_full'] = []
        dict_d['mean_normFIM_spectra_cut'] = []
        dict_d['std_normFIM_spectra_cut'] = []
        dict_d['min_normFIM_spectra_cut'] = []
        dict_d['max_normFIM_spectra_cut'] = []
        dict_d['norm_eff_dim_cut'] = []
        dict_unbiased_all[max_freq] = dict_d
    for filename in listallfiles:
        res = re.findall((name_0 + name_1 + '(\S+)' + name_2 + name_3 + '(\S+).pkl'), filename)
        max_freq = res[0][0]
        max_freq = int(max_freq)
        nm = res[0][1]
        nm = int(nm)
        path_file = os.path.join(results_folder, filename)
        with open(path_file, 'rb') as f:
            dict_model = pickle.load(f)
        dict_d = dict_unbiased_all[max_freq]
        dict_d['delta_minMSE_mean'].append(np.mean(dict_model['delta_minMSE_full_m_cut']))
        dict_d['delta_minMSE_std'].append(np.std(dict_model['delta_minMSE_full_m_cut']))
        dict_d['delta_minMSE_min'].append(np.min(dict_model['delta_minMSE_full_m_cut']))
        dict_d['delta_minMSE_max'].append(np.max(dict_model['delta_minMSE_full_m_cut']))
        dict_d['MSE_full_mean'].append(dict_model['MSE_full_mean'])
        dict_d['MSE_full_std'].append(dict_model['MSE_full_std'])
        dict_d['MSE_full_min'].append(dict_model['MSE_full_min'])
        dict_d['MSE_full_max'].append(dict_model['MSE_full_max'])
        dict_d['MSE_cut_mean'].append(dict_model['MSE_cut_mean'])
        dict_d['MSE_cut_std'].append(dict_model['MSE_cut_std'])
        dict_d['MSE_cut_min'].append(dict_model['MSE_cut_min'])
        dict_d['MSE_cut_max'].append(dict_model['MSE_cut_max'])
        dict_d['mean_normFIM_spectra_full'].append(dict_model['mean_normFIM_spectra_full'])
        dict_d['std_normFIM_spectra_full'].append(dict_model['std_normFIM_spectra_full'])
        dict_d['min_normFIM_spectra_full'].append(dict_model['min_normFIM_spectra_full'])
        dict_d['max_normFIM_spectra_full'].append(dict_model['max_normFIM_spectra_full'])
        dict_d['norm_eff_dim_full'].append(dict_model['norm_eff_dim_full'])
        dict_d['mean_normFIM_spectra_cut'].append(dict_model['mean_normFIM_spectra_cut'])
        dict_d['std_normFIM_spectra_cut'].append(dict_model['std_normFIM_spectra_cut'])
        dict_d['min_normFIM_spectra_cut'].append(dict_model['min_normFIM_spectra_cut'])
        dict_d['max_normFIM_spectra_cut'].append(dict_model['max_normFIM_spectra_cut'])
        dict_d['norm_eff_dim_cut'].append(dict_model['norm_eff_dim_cut'])
        dict_unbiased_all[max_freq] = dict_d
    print('Finished unbiased for Ntrain ', n_train)
    print(' ')

    name_end_extr = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + 
                     '_DimBasisParam' + name_dim_basis_param + '_NtrainPerFeat' + str(n_train) + 
                     '_batch' + batch_size_name + '_lr' + learning_rate_name + '_cutoff' + cutoff_name + 
                     '_decayexp' + decay_exp_name)
    
    filename = 'extracted_results_FULLbiased' + name_end_extr + '.pkl'
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'wb') as f:
        pickle.dump(dict_fullbiased_all, f)
    
    filename = 'extracted_results_UNbiased' + name_end_extr + '.pkl'
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'wb') as f:
        pickle.dump(dict_unbiased_all, f)

Loads the aggregated `extracted_results_FULLbiased`/`extracted_results_UNbiased` pickles for each available Ntrain and, for every MaxFreq value, averages `Delta_{f-c}MSE_min` over model draws, building the biased-vs-unbiased scaling trend of `Delta_{f-c}MSE_min` with MaxFreq.

In [ ]:
no_train_vec = all_n_train

dict_allNtrain = dict()
for no_train in no_train_vec:
    name_end_extr = ('_BondDim' + name_bond_dim + '_Nfeatures' + name_no_features + 
                     '_DimBasisParam' + name_dim_basis_param + '_NtrainPerFeat' + str(no_train) + 
                     '_batch' + batch_size_name + '_lr' + learning_rate_name + '_cutoff' + cutoff_name + 
                     '_decayexp' + decay_exp_name)
        
    filename = 'extracted_results_FULLbiased' + name_end_extr + '.pkl'
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'rb') as f:
        dict_fullbiased_all = pickle.load(f)
    
    filename = 'extracted_results_UNbiased' + name_end_extr + '.pkl'
    path_file = os.path.join(results_folder, filename)
    with open(path_file, 'rb') as f:
        dict_unbiased_all = pickle.load(f)
    
    dMSE_biased = []
    dMSE_unbiased = []
    for max_freq in max_frequency_vec:
        dict_nopars_biased = dict_fullbiased_all[max_freq]
        deltaMSE_biased = np.asarray(dict_nopars_biased['delta_minMSE_mean'])
        dMSE_biased.append(np.mean(deltaMSE_biased))
        dict_nopars_unbiased = dict_unbiased_all[max_freq]
        deltaMSE_unbiased = np.asarray(dict_nopars_unbiased['delta_minMSE_mean'])
        dMSE_unbiased.append(np.mean(deltaMSE_unbiased))
    yy_b = np.asarray(dMSE_biased)
    yy_u = np.asarray(dMSE_unbiased)
    dict_Ntrain = dict()
    dict_Ntrain['deltaMSE_biased'] = yy_b
    dict_Ntrain['deltaMSE_unbiased'] = yy_u
    dict_allNtrain[no_train] = dict_Ntrain

Plots `Delta_{f-c}MSE_min` (mean over model draws) against the input-basis dimension D = 2*MaxFreq - 1 for the unbiased data-generating case (the biased scatter is computed but plotted with a commented-out line), with points colored by Ntrain, reproducing the paper's MaxFreq-scaling comparison of training performance between the full and cutoff models.

In [ ]:
fs = 28
figsize = (8,4)

cmap = matplotlib.colormaps['cividis']

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

cmap = matplotlib.colormaps['viridis']
xx = []
yy_b = []
yy_u = []
zz = []
for no_train in no_train_vec:
    dict_Ntrain = dict_allNtrain[no_train]
    y_b = dict_Ntrain['deltaMSE_biased']
    y_u = dict_Ntrain['deltaMSE_unbiased']
    for i in range(len(y_b)):
        xx.append(max_frequency_vec[i])
        yy_b.append(y_b[i])
        yy_u.append(y_u[i])
        zz.append(no_train)
xx = np.asarray(xx)
yy_b = np.asarray(yy_b)
yy_u = np.asarray(yy_u)
zz = np.asarray(zz)

#fA = ax.scatter(2*xx-1, yy_b, marker='o', s=70, c=zz, cmap=cmap)
fA = ax.scatter(2*xx-1, yy_u, marker='o', s=70, c=zz, cmap=cmap)
cbar = fig.colorbar(fA, ax=ax, extend='both')
cbar.ax.set_ylabel(r'$n_{\mathrm{train}}$', fontsize=fs)

#ax.legend(fontsize=22)
ax.set_xlabel(r'$D$', fontsize=fs)
ax.set_ylabel(r'$\Delta_{\mathrm{f-c}}\,\mathrm{MSE}_{\mathrm{min}}$', fontsize=fs)
#ax.set_xscale('log')
#ax.set_yscale('log')
#ax.set_ylim([1.0e-12, 1.0e03])

Saves the resulting figure to disk in PNG and PDF format.

In [ ]:
fig.savefig('DeltaMSEmin_unbiased_vs_L_Dloc3_chi120_M50_R2.png', bbox_inches='tight')
fig.savefig('DeltaMSEmin_unbiased_vs_L_Dloc3_chi120_M50_R2.pdf', bbox_inches='tight')
#fig.savefig('DeltaMSEmin_fullbiased_vs_D_Dloc3_chi120_M50_R2.png', bbox_inches='tight')
#fig.savefig('DeltaMSEmin_fullbiased_vs_D_Dloc3_chi120_M50_R2.pdf', bbox_inches='tight')